## Visualization of random image
This notebook is to visualization of random images from each category of the images to visualize the image, its ground truth, its predicted mask and the overlay of the prediction. Green pixels are correctly predicted, red are pixels which are FP (not present but predicted as present) and blue pixels are FN (actually present but not predicted). Some metrics related to each image is also given as output. 

Expected directory structure:
```
root directory
    |--> root image directory
    |            |--> category 1 (this is also the text prompt)
    |            |--> category 2
    |
    |--> root mask directory
                 |--> category 1
                 |--> category 2
```


In [ ]:
import torch
import numpy as np
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm
from sam3 import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

In [ ]:
def load_model(bpe_path: str, sam3_checkpoint: Path, finetuned_checkpoint: Path, 
               device: torch.device = None) -> torch.nn.Module:
    """Load SAM3 model with fine-tuned weights.
     Args:
         bpe_path: Path to the BPE tokenizer file.
         sam3_checkpoint: Path to the original SAM3 model checkpoint.
         finetuned_checkpoint: Path to the fine-tuned model checkpoint.
         device: Optional torch device to load the model on. If None, it will automatically use CUDA if available.
     Returns:
         Loaded SAM3 model with fine-tuned weights."""
    
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    model = build_sam3_image_model(bpe_path=bpe_path, checkpoint_path=sam3_checkpoint)
    finetuned_weights = torch.load(finetuned_checkpoint, map_location=device)
    model.load_state_dict(finetuned_weights['model'])
    model.eval()
    
    print("Model loaded")
    return model

In [ ]:
def find_matching_mask(image_path: Path, masks_dir: Path) -> Path:
    """Find the corresponding ground truth mask for an image.
     Args:
         image_path: Path to the input image.
         masks_dir: Directory containing ground truth masks.
     Returns:
         Path to the matching ground truth mask.
     Raises:
         FileNotFoundError: If no matching mask is found for the given image."""
    
    stem = image_path.stem
    for ext in ['.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp']:
        mask_path = masks_dir / f"{stem}{ext}"
        if mask_path.exists():
            return mask_path
    raise FileNotFoundError(f"No mask found for {image_path.name} in {masks_dir}")

In [ ]:
def load_ground_truth_mask(mask_path: Path) -> np.ndarray:
    """Load ground truth mask from file.
     Args:
         mask_path: Path to the ground truth mask file.
     Returns:
         Ground truth mask as a binary numpy array."""
    
    mask = np.array(Image.open(mask_path).convert('L'))
    return (mask > 127).astype(np.uint8)

In [ ]:
def get_predicted_mask(model, image: Image.Image, prompt: str, 
                       confidence_threshold: float = 0.5) -> np.ndarray:
    """Run inference and get combined predicted mask.
     Args:
         model: The loaded SAM3 model for inference.
         image: Input image as a PIL Image.
         prompt: Text prompt to guide the segmentation.
         confidence_threshold: Confidence threshold for mask prediction.
     Returns:
         Combined predicted mask as a binary numpy array."""
    processor = Sam3Processor(model, confidence_threshold=confidence_threshold)
    inference_state = processor.set_image(image)
    processor.reset_all_prompts(inference_state)
    inference_state = processor.set_text_prompt(state=inference_state, prompt=prompt)
    
    masks = inference_state.get('masks', [])
    
    if len(masks) == 0:
        return np.zeros((image.height, image.width), dtype=np.uint8)
    
    combined_mask = np.zeros((image.height, image.width), dtype=bool)
    for mask in masks:
        if isinstance(mask, torch.Tensor):
            mask = mask.cpu().numpy()
        if mask.ndim == 3:
            mask = mask.squeeze()
        combined_mask = np.logical_or(combined_mask, mask > 0.5)
    
    return combined_mask.astype(np.uint8)

In [ ]:
def compute_foreground_iou(pred_mask: np.ndarray, gt_mask: np.ndarray) -> dict:
    """
    Compute Foreground IoU between predicted and ground truth masks.
    
    Foreground IoU = TP / (TP + FP + FN)
    
    Args:
        pred_mask: Predicted binary mask (H, W) - 1 for foreground, 0 for background
        gt_mask: Ground truth binary mask (H, W) - 1 for foreground, 0 for background
    
    Returns:
        Dictionary with foreground_iou, TP, FP, FN, TN pixel counts
    """
    pred_mask = pred_mask.astype(bool)
    gt_mask = gt_mask.astype(bool)
    
    # True Positive: Both predict and GT are foreground
    TP = np.logical_and(pred_mask, gt_mask).sum()
    
    # False Positive: Predicted foreground but GT is background
    FP = np.logical_and(pred_mask, ~gt_mask).sum()
    
    # False Negative: Predicted background but GT is foreground
    FN = np.logical_and(~pred_mask, gt_mask).sum()
    
    # True Negative: Both predict and GT are background
    TN = np.logical_and(~pred_mask, ~gt_mask).sum()
    
    # Foreground IoU
    denominator = TP + FP + FN
    if denominator == 0:
        fg_iou = 1.0  # No foreground in either - perfect match
    else:
        fg_iou = TP / denominator
    
    return {
        'foreground_iou': fg_iou,
        'TP': int(TP),
        'FP': int(FP),
        'FN': int(FN),
        'TN': int(TN),
        'precision': TP / (TP + FP) if (TP + FP) > 0 else 0,
        'recall': TP / (TP + FN) if (TP + FN) > 0 else 0,
    }

In [ ]:
def visualize_foreground_iou(image_path: str, model, prompt: str,
                              masks_dir: Path,
                              confidence_threshold: float = 0.5):
    """
    Visualize a single image with Foreground IoU metrics.
    Args:
        image_path: Path to the input image.
        model: The loaded SAM3 model for inference.
        prompt: Text prompt to guide the segmentation.
        masks_dir: Directory containing ground truth masks.
        confidence_threshold: Confidence threshold for mask prediction.
    """
    image_path = Path(image_path)
    
    # Load image and masks
    image = Image.open(image_path).convert('RGB')
    mask_path = find_matching_mask(image_path, masks_dir)
    gt_mask = load_ground_truth_mask(mask_path)
    
    with torch.no_grad():
        pred_mask = get_predicted_mask(model, image, prompt, confidence_threshold)
    
    if pred_mask.shape != gt_mask.shape:
        pred_mask = np.array(
            Image.fromarray(pred_mask).resize(
                (gt_mask.shape[1], gt_mask.shape[0]), Image.NEAREST
            )
        )
    
    # Compute metrics
    fg_metrics = compute_foreground_iou(pred_mask, gt_mask)
    
    # Create figure
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    
    axes[0].imshow(image)
    axes[0].set_title('Original Image')
    axes[0].axis('off')
    
    axes[1].imshow(gt_mask, cmap='gray')
    axes[1].set_title(f'Ground Truth\n({gt_mask.sum():,} fg pixels)')
    axes[1].axis('off')
    
    axes[2].imshow(pred_mask, cmap='gray')
    axes[2].set_title(f'Predicted\n({pred_mask.sum():,} fg pixels)')
    axes[2].axis('off')
    
    # Overlay
    overlay = np.zeros((*gt_mask.shape, 3), dtype=np.uint8)
    overlay[np.logical_and(pred_mask, gt_mask)] = [0, 255, 0]       # TP (green)
    overlay[np.logical_and(pred_mask, ~gt_mask.astype(bool))] = [255, 0, 0]   # FP (red)
    overlay[np.logical_and(~pred_mask.astype(bool), gt_mask)] = [0, 0, 255]   # FN (blue)
    
    axes[3].imshow(overlay)
    axes[3].set_title(f'FG IoU: {fg_metrics["foreground_iou"]:.3f}')
    axes[3].axis('off')
    
    plt.suptitle(f'{image_path.name}', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print(f" Image: {image_path.name}")
    print(f" (Green)True Positive={fg_metrics['TP']:,} pixels | (Red)False Positive={fg_metrics['FP']:,} pixels | (Blue)False Negative={fg_metrics['FN']:,} pixels")
    print(f" Foreground IoU: {fg_metrics['foreground_iou']:.4f} | Precision: {fg_metrics['precision']:.4f} | Recall: {fg_metrics['recall']:.4f}")


In [ ]:
def visualize_image(image_name: str):
    """
    Visualize any image by its filename.
    
    Args:
        image_name: Name of the image file (e.g., 'image_001.png')
    """
    
    image_path = TEST_IMAGES_PATH / image_name
    visualize_foreground_iou(
        image_path=image_path,
        model=model,
        prompt=TEXT_PROMPT,
        masks_dir=GT_MASKS_PATH,
        confidence_threshold=CONFIDENCE_THRESHOLD
    )

In [ ]:
# loop through each category in the root image directory and visualize N random images from each category
def visualize_random_samples(test_images_path: Path, model, gt_masks_path: Path,
                              number_of_samples_per_category: int = 2,
                              confidence_threshold: float = 0.5, seed=None):
    """Main loop to visualize random samples from each category in the test images directory.
    Args:
        test_images_path: Root directory containing subdirectories for each category of test images.
        model: The loaded SAM3 model to use for inference.
        gt_masks_path: Root directory containing ground truth masks, structured similarly to test_images_path.
        number_of_samples_per_category: Number of random images to visualize from each category.
        confidence_threshold: Confidence threshold for mask prediction.
        seed: Optional random seed for reproducibility.
    """
    if seed is not None:
        np.random.seed(seed)
    for category_dir in test_images_path.iterdir():
        prompt = category_dir.name
        if category_dir.is_dir():
            image_files = list(category_dir.glob('*.*'))
            if len(image_files) == 0:
                continue
            selected_images = np.random.choice(image_files, size=min(number_of_samples_per_category, len(image_files)), replace=False)
            for image_path in selected_images:
                # Derive the category mask directory by mirroring the images folder structure under masks root
                mask_category_dir = Path(str(image_path.parent).replace(str(test_images_path), str(gt_masks_path)))
                visualize_foreground_iou(
                    image_path=image_path,
                    model=model,
                    prompt=prompt,
                    masks_dir=mask_category_dir,
                    confidence_threshold=confidence_threshold
                )


In [ ]:
# Path to  test images root directory (contains subdirectories for each category)
TEST_IMAGES_PATH = Path('/home/hpc_storage/data/projects/userA/sam3_data/multi_test/images')

# Path to ground truth masks root directory (mirrors the structure of test images)
GT_MASKS_PATH = Path('/home/hpc_storage/data/projects/userA/sam3_data/multi_test/masks')

# Model paths
BPE_PATH = '/home/hpc_storage/data/projects/userA/sam3/sam3/assets/bpe_simple_vocab_16e6.txt.gz'
SAM3_CHECKPOINT = Path('/home/hpc_storage/data/projects/userA/sam3/models--facebook--sam3/sam3.pt')
FINETUNED_CHECKPOINT = Path('/home/hpc_storage/data/projects/userA/sam3_data/experiments/testing_2_3pins_1crack/checkpoints/checkpoint_10.pt')

# Inference settings
CONFIDENCE_THRESHOLD = 0.5
seed = 42
number_of_samples_per_category = 2

model = load_model(BPE_PATH, SAM3_CHECKPOINT, FINETUNED_CHECKPOINT)
visualize_random_samples(
    test_images_path=TEST_IMAGES_PATH,
    model=model,
    gt_masks_path=GT_MASKS_PATH,
    number_of_samples_per_category=number_of_samples_per_category,
    confidence_threshold=CONFIDENCE_THRESHOLD,
    seed=seed
)